# Data Cleaning con Pandas

> Pandas es una librería de Python escrita como extensión de NumPy para manipulación y análisis de datos.

Sitio oficial: [pandas.python.org](https://pandas.pydata.org/)
Documentación Oficial: [pandas.pydata.org/pandas-docs/stable/](https://pandas.pydata.org/pandas-docs/stable/)

In [1]:
import pandas as pd

## DataFrame Querying

En este notebook hablaremos sobre cómo consultar DataFrames. El primer paso es comprender el enmascaramiento booleano. El enmascaramiento booleano es fundamental para realizar consultas rápidas y eficientes en NumPy y Pandas, y es análogo al enmascaramiento de bits utilizado en otras áreas de la informática. Al finalizar esta lección, comprenderás cómo funciona el enmascaramiento booleano y cómo aplicarlo a un DataFrame para obtener los datos que te interesan.

Una **máscara booleana** es un array que puede ser unidimensional, como una serie, o bidimensional, como un DataFrame, donde cada valor del array es verdadero o falso. Este array se superpone a la estructura de datos que estamos consultando. Cualquier celda con un valor verdadero se incluirá en el resultado final, mientras que cualquier celda con un valor falso no.

In [2]:
df = pd.read_csv('../data/admission-predict.csv', index_col=0)
df.columns = [x.lower().strip().replace(' ', '_') for x in df.columns]

df.head()

,gre_score,toefl_score,university_rating,sop,lor,cgpa,research,chance_of_admit
Serial No.,,,,,,,,
1,337,118,4,4.5,4.5,9.65,1,0.92
2,324,107,4,4.0,4.5,8.87,1,0.76
3,316,104,3,3.0,3.5,8.00,1,0.72
4,322,110,3,3.5,2.5,8.67,1,0.80
5,314,103,2,2.0,3.0,8.21,0,0.65


Las máscaras booleanas se crean aplicando operadores directamente a los objetos Series o DataFrame de pandas.

### Ejemplo 1


Por ejemplo, en nuestro conjunto de datos de admisión a posgrado, podríamos estar interesados en ver solo a los estudiantes con una probabilidad de admisión superior a 0.7.

Para crear una máscara booleana para esta consulta, proyectamos la columna de probabilidad de admisión usando el operador de indexación y aplicamos el operador "mayor que" con un valor de comparación de 0.7. Esto equivale a aplicar el operador de comparación "mayor que", cuyos resultados se devuelven como una Serie booleana. La Serie resultante se indexa de manera que el valor de cada celda sea verdadero o falso, dependiendo de si un estudiante tiene una probabilidad de admisión superior a 0.7.

In [3]:
admit_mask = df['chance_of_admit'] > 0.7
admit_mask

Serial No.
1       True
2       True
3       True
4       True
5      False
       ...  
396     True
397     True
398     True
399    False
400     True
Name: chance_of_admit, Length: 400, dtype: bool

El resultado de aplicar un operador de comparación es una máscara booleana: obtenemos valores verdaderos o falsos según el resultado de la comparación.

Internamente, pandas aplica el operador de comparación especificado mediante vectorización (de forma eficiente y en paralelo) a todos los valores del array especificado. El resultado es una Serie (ya que solo se opera sobre una columna) rellena con valores verdaderos o falsos, que es lo que devuelve el operador de comparación.


Ahora, podemos superponerla a los datos para "ocultar" los datos que no queremos, representados por todos los valores falsos. Esto se hace utilizando la función `.where()` en el DataFrame original.


In [4]:
df.where(admit_mask).head(10)

,gre_score,toefl_score,university_rating,sop,lor,cgpa,research,chance_of_admit
Serial No.,,,,,,,,
1,337.0,118.0,4.0,4.5,4.5,9.65,1.0,0.92
2,324.0,107.0,4.0,4.0,4.5,8.87,1.0,0.76
3,316.0,104.0,3.0,3.0,3.5,8.00,1.0,0.72
4,322.0,110.0,3.0,3.5,2.5,8.67,1.0,0.80
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,330.0,115.0,5.0,4.5,3.0,9.34,1.0,0.90
7,321.0,109.0,3.0,3.0,4.0,8.20,1.0,0.75
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Observamos que el dataframe resultante conserva los valores indexados originales y solo se retuvieron los datos que cumplieron la condición. Todas las filas que no cumplieron la condición contienen valores NaN, pero estas filas no se eliminaron del conjunto de datos.

Si no queremos eliminar los valores NaN, usar la función `dropna()`.

In [5]:
df.where(admit_mask).dropna().head(10)

,gre_score,toefl_score,university_rating,sop,lor,cgpa,research,chance_of_admit
Serial No.,,,,,,,,
1,337.0,118.0,4.0,4.5,4.5,9.65,1.0,0.92
2,324.0,107.0,4.0,4.0,4.5,8.87,1.0,0.76
3,316.0,104.0,3.0,3.0,3.5,8.00,1.0,0.72
4,322.0,110.0,3.0,3.5,2.5,8.67,1.0,0.80
6,330.0,115.0,5.0,4.5,3.0,9.34,1.0,0.90
7,321.0,109.0,3.0,3.0,4.0,8.20,1.0,0.75
12,327.0,111.0,4.0,4.0,4.5,9.00,1.0,0.84
13,328.0,112.0,4.0,4.0,4.5,9.10,1.0,0.78
23,328.0,116.0,5.0,5.0,5.0,9.50,1.0,0.94


> Nota: En el DataFrame devuelto se han eliminado todas las filas con `NaN`. Observe los índices. Incluye del uno al cuatro, el seis y siete..., pero no el cinco, ni el ocho...

A pesar de ser muy útil, `where()` no se usa con frecuencia. En cambio, los desarrolladores de pandas crearon una sintaxis abreviada que combina `where()` y `dropna()`, realizando ambas operaciones a la vez. Y, como suele suceder, ¡simplemente sobrecargaron el operador de indexación para lograrlo!

In [6]:
df[df['chance_of_admit'] > 0.7].head(10)

,gre_score,toefl_score,university_rating,sop,lor,cgpa,research,chance_of_admit
Serial No.,,,,,,,,
1,337,118,4,4.5,4.5,9.65,1,0.92
2,324,107,4,4.0,4.5,8.87,1,0.76
3,316,104,3,3.0,3.5,8.00,1,0.72
4,322,110,3,3.5,2.5,8.67,1,0.80
6,330,115,5,4.5,3.0,9.34,1,0.90
7,321,109,3,3.0,4.0,8.20,1,0.75
12,327,111,4,4.0,4.5,9.00,1,0.84
13,328,112,4,4.0,4.5,9.10,1,0.78
23,328,116,5,5.0,5.0,9.50,1,0.94


### Ejemplo 2

In [7]:
# Se puede enviar el nombre de la columna como string
df["gre_score"].head()

Serial No.
1    337
2    324
3    316
4    322
5    314
Name: gre_score, dtype: int64

In [8]:
# También podemos enviar una lista de nombres de columnas
df[["gre_score", "toefl_score"]].head()

,gre_score,toefl_score
Serial No.,,
1,337,118
2,324,107
3,316,104
4,322,110
5,314,103


In [9]:
# O podemos enviar una mascara booleana
df[df["gre_score"] > 320].head()

,gre_score,toefl_score,university_rating,sop,lor,cgpa,research,chance_of_admit
Serial No.,,,,,,,,
1,337,118,4,4.5,4.5,9.65,1,0.92
2,324,107,4,4.0,4.5,8.87,1,0.76
4,322,110,3,3.5,2.5,8.67,1,0.80
6,330,115,5,4.5,3.0,9.34,1,0.90
7,321,109,3,3.0,4.0,8.20,1,0.75


> Nota: Cada uno de estos imita la funcionalidad de `.loc()` o `.where().dropna().`

### Ejemplo 3

Combinar varias máscaras booleanas (incluir múltiples criterios).

En Pandas, debes usar operadores bit a bit en lugar de los operadores lógicos de Python:
- En lugar de and, usa &
- En lugar de or, usa |
- En lugar de not, usa ~

Importante: Cuando uses estos operadores, debes envolver cada condición entre paréntesis debido a la precedencia de operadores en Python.

Entonces, el enmascaramiento se hace con "&" (si ambas máscaras deben ser verdaderas para que un valor verdadero aparezca en la máscara final), o con "|" (si solo una debe ser verdadera).

Por ejemplo, si quieres combinar dos series booleanas con "&", ¿cómo puedes hacerlo?

In [10]:
(df['chance_of_admit'] > 0.7) & (df['university_rating'] == 5)

Serial No.
1      False
2      False
3      False
4      False
5      False
       ...  
396    False
397    False
398    False
399    False
400    False
Length: 400, dtype: bool

Otra forma de hacerlo es eliminar por completo el operador de comparación y, en su lugar, utilizar las funciones integradas que imitan este enfoque.

In [11]:
df['chance_of_admit'].gt(0.7) & df['chance_of_admit'].lt(0.9)

Serial No.
1      False
2       True
3       True
4       True
5      False
       ...  
396     True
397     True
398    False
399    False
400    False
Name: chance_of_admit, Length: 400, dtype: bool

Estas funciones están integradas en los objetos Series y DataFrame, por lo que también puedes encadenarlas, lo que da como resultado la misma solución sin necesidad de operadores visuales.

Tú decides qué opción te parece mejor.

In [12]:
df['chance_of_admit'].gt(0.7).lt(0.9)

Serial No.
1      False
2      False
3      False
4      False
5       True
       ...  
396    False
397    False
398    False
399     True
400    False
Name: chance_of_admit, Length: 400, dtype: bool

## Conclusiones:

Aprendimos a consultar dataframes usando máscaras booleanas, una técnica fundamental y de uso frecuente en el mundo de la ciencia de datos. Con el enmascaramiento booleano, podemos seleccionar datos según los criterios que deseemos y, sinceramente, la usarás en todas partes.

Debes saber leer y escribir todo esto, y comprender las implicaciones del método que elijas. Diría que al menos el 50 % del trabajo de limpieza de datos implica consultar dataframes.